In [0]:
import pandas as pd

In [0]:
query = """
SELECT 
    fs.order_date,
    fs.product_fk,
    fs.territory_fk,
    fs.order_quantity,
    fs.unit_price,
    fs.total_due,
    dp.product_name,
    dp.product_category_name,
    dt.country_region_name,
    ds.store_name,
    ds.business_entity_id as store_id
FROM ted_dev.marts.fact_sales fs
JOIN ted_dev.marts.dim_product dp ON fs.product_fk = dp.product_pk
JOIN ted_dev.marts.dim_territory dt ON fs.territory_fk = dt.territory_pk
LEFT JOIN ted_dev.marts.dim_store ds ON fs.sales_person_fk = ds.sales_person_id
ORDER BY fs.order_date
"""

df = spark.sql(query).toPandas()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dados carregados: {len(df):,} registros")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")
print(f"Produtos únicos: {df['product_fk'].nunique()}")
print(f"Lojas físicas: {df['store_id'].nunique()}")

monthly_data = df.groupby([
    'product_fk', 
    'store_id',
    pd.Grouper(key='order_date', freq='M')
]).agg({
    'order_quantity': 'sum',
    'unit_price': 'mean',
    'total_due': 'sum',
    'product_name': 'first',
    'store_name': 'first',
    'country_region_name': 'first'
}).reset_index()

monthly_data = monthly_data.dropna(subset=['store_id'])

In [0]:
display(monthly_data)